In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:
class PatchDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        # Binary mapping
        self.label_map = {
            "normal": 0,
            "hcc": 1,
            "chol": 1
        }

        for cls in ["normal", "hcc", "chol"]:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.exists(cls_dir):
                continue

            for slide_id in os.listdir(cls_dir):
                slide_path = os.path.join(cls_dir, slide_id)
                if not os.path.isdir(slide_path):
                    continue

                for fname in os.listdir(slide_path):
                    if fname.endswith(".png"):
                        self.samples.append(
                            (os.path.join(slide_path, fname),
                             self.label_map[cls])
                        )

        print(f"Loaded {len(self.samples)} patches from {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.long)


In [4]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [5]:
train_dataset = PatchDataset(
    root_dir="TRAIN",
    transform=train_transform
)

test_dataset = PatchDataset(
    root_dir="TEST",
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)


Loaded 111300 patches from TRAIN
Loaded 28000 patches from TEST


In [6]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace classifier
model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)


/home/jupyter-238w1a05a8/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [7]:
class_weights = torch.tensor([1.5, 1.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [8]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


In [9]:
def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)

            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    return np.array(y_true), np.array(y_pred)


In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    loss, acc = train_epoch(model, train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss:.4f} | Train Acc: {acc:.4f}")


In [ ]:
y_true, y_pred = evaluate(model, test_loader)

print("Test Accuracy:", accuracy_score(y_true, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=["Normal", "Cancer"],
    digits=4,
    zero_division=0
))


In [ ]:
def get_slide_ids(root_dir):
    slides = set()
    for cls in ["normal", "hcc", "chol"]:
        cls_dir = os.path.join(root_dir, cls)
        if not os.path.exists(cls_dir):
            continue
        for slide_id in os.listdir(cls_dir):
            slides.add(slide_id)
    return slides

train_slides = get_slide_ids("TRAIN")
test_slides  = get_slide_ids("TEST")

print("Train slides:", len(train_slides))
print("Test slides :", len(test_slides))
print("Overlap     :", len(train_slides & test_slides))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd


In [ ]:
class_names = ["Normal", "Cancer"]

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=False,
    linewidths=0.5
)

plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title("Confusion Matrix – Patch-Level CNN (Model-A)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".4f",
    cmap="YlGnBu",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=True,
    linewidths=0.5
)

plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title("Normalized Confusion Matrix – Patch-Level CNN", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4,
    output_dict=True
)

df_report = pd.DataFrame(report).transpose()

plt.figure(figsize=(8, 4))
sns.heatmap(
    df_report.iloc[:-1, :3],
    annot=True,
    cmap="Blues",
    fmt=".4f",
    cbar=False
)

plt.title("Classification Report – Patch-Level CNN", fontsize=14)
plt.tight_layout()
plt.show()

df_report


In [ ]:
torch.save(model.state_dict(), "model_A11_patch_binary.pth")
print("Model-A (patch CNN) saved successfully.")
